# 量子コンピュータによる暗号脅威：Shorのアルゴリズム

## なぜ耐量子暗号 (PQC) が必要か？

現代の公開鍵暗号 (RSA, ECC) は以下の数学的困難性に基づいています：
- **RSA**: 大きな数の素因数分解の困難性
- **ECC/ECDH**: 楕円曲線離散対数問題の困難性

Peter Shor (1994) が考案したアルゴリズムは、**量子コンピュータ上で多項式時間**でこれらを解けることを証明しました。

```
古典コンピュータ: RSA-2048 の解読 → 宇宙の年齢より長い時間
量子コンピュータ: RSA-2048 の解読 → 数時間〜数日 (理論値)
```

In [ ]:
# 必要なライブラリのインポート
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import QFT
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from math import gcd
from fractions import Fraction

print('Qiskit環境の確認完了')

## 1. Shorのアルゴリズムの原理

### アルゴリズムの流れ

```
素因数分解問題: N = p × q を解く

STEP 1: a を N とコプライムになるよう選ぶ (gcd(a,N)=1)
STEP 2: 量子回路で f(x) = a^x mod N の周期 r を求める
STEP 3: r が偶数なら p = gcd(a^(r/2)+1, N) で因数を得る
STEP 4: N = p × q が完成
```

### 量子位相推定 (QPE) による周期発見

In [ ]:
def classical_period_finding(a, N):
    """古典的な周期発見（比較用）"""
    x = 1
    for r in range(1, N):
        x = (x * a) % N
        if x == 1:
            return r
    return None

def classical_shor(N):
    """古典シミュレーションによるShorアルゴリズム"""
    if N % 2 == 0:
        return 2, N // 2
    
    for a in range(2, N):
        g = gcd(a, N)
        if g != 1:
            return g, N // g
        
        # 周期発見（量子部分を古典でシミュレート）
        r = classical_period_finding(a, N)
        
        if r is None or r % 2 != 0:
            continue
        
        # 因数の計算
        p = gcd(pow(a, r // 2) + 1, N)
        q = gcd(pow(a, r // 2) - 1, N)
        
        if p not in [1, N] and q not in [1, N]:
            return p, q
    
    return None, None

# 小さな数で動作確認
test_cases = [15, 21, 35, 77]
print('Shorアルゴリズムの動作確認（古典シミュレーション）')
print('=' * 50)
for N in test_cases:
    p, q = classical_shor(N)
    print(f'N = {N:4d} → {p} × {q} = {p*q} ✓' if p and q else f'N = {N:4d} → 因数発見失敗')

## 2. Qiskitで実装するShorのアルゴリズム（N=15）

N=15 (= 3 × 5) を量子回路で素因数分解する実装例です。

In [ ]:
def c_amod15(a, power):
    """制御付き a^2^power mod 15 ゲート"""
    if a not in [2, 7, 8, 11, 13]:
        raise ValueError(f'a={a} は N=15 とコプライムではありません')
    
    U = QuantumCircuit(4)
    for _ in range(power):
        if a in [2, 13]:
            U.swap(0, 1)
            U.swap(1, 2)
            U.swap(2, 3)
        elif a in [7, 8]:
            U.swap(2, 3)
            U.swap(1, 2)
            U.swap(0, 1)
        elif a == 11:
            U.swap(1, 3)
            U.swap(0, 2)
        if a in [7, 11, 13]:
            for q in range(4):
                U.x(q)
    
    U = U.to_gate()
    U.name = f'{a}^{power} mod 15'
    c_U = U.control()
    return c_U

def quantum_phase_estimation_circuit(a, n_count=8):
    """量子位相推定回路の構築"""
    # カウントレジスタ: n_count量子ビット
    # 作業レジスタ: 4量子ビット (|1> で初期化)
    qc = QuantumCircuit(n_count + 4, n_count)
    
    # 作業レジスタを |1> に初期化
    qc.x(n_count)
    
    # カウントレジスタにアダマールゲート
    for q in range(n_count):
        qc.h(q)
    
    # 制御ユニタリ操作
    for q in range(n_count):
        qc.append(c_amod15(a, 2**q), [q] + list(range(n_count, n_count + 4)))
    
    # 逆量子フーリエ変換
    qc.append(QFT(n_count, inverse=True), range(n_count))
    
    # 測定
    qc.measure(range(n_count), range(n_count))
    
    return qc

# 回路の構築
a = 7  # テスト用の a (gcd(7, 15) = 1)
n_count = 8
qc = quantum_phase_estimation_circuit(a, n_count)

print(f'量子回路の情報:')
print(f'  量子ビット数: {qc.num_qubits}')
print(f'  古典ビット数: {qc.num_clbits}')
print(f'  ゲート数: {qc.size()}')
print(f'  回路の深さ: {qc.depth()}')

In [ ]:
# シミュレーションの実行
simulator = AerSimulator()
job = simulator.run(qc, shots=2048)
counts = job.result().get_counts()

# 上位結果の表示
print(f'測定結果 (上位10件):')
sorted_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:10]
for bitstring, count in sorted_counts:
    phase = int(bitstring, 2) / 2**n_count
    frac = Fraction(phase).limit_denominator(15)
    print(f'  {bitstring} ({count:4d}回) → 位相={phase:.4f} → r候補={frac.denominator}')

# ヒストグラムの表示
plot_histogram(counts, figsize=(12, 5), title=f'Shorアルゴリズム: a={a}, N=15 の測定結果')
plt.tight_layout()
plt.savefig('shor_result.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def extract_factors_from_measurement(measured_int, a, N, n_count):
    """測定結果から因数を抽出"""
    phase = measured_int / 2**n_count
    frac = Fraction(phase).limit_denominator(N)
    r = frac.denominator
    
    if r % 2 != 0:
        return None, None, r
    
    p = gcd(pow(a, r // 2) + 1, N)
    q = gcd(pow(a, r // 2) - 1, N)
    
    return p, q, r

# 全測定結果を解析
print(f'因数解析結果 (a={a}, N={N}):')
print('=' * 60)
N = 15
found_factors = set()

for bitstring, count in sorted_counts:
    measured_int = int(bitstring, 2)
    if measured_int == 0:
        continue
    
    p, q, r = extract_factors_from_measurement(measured_int, a, N, n_count)
    
    if p and q and p not in [1, N] and q not in [1, N]:
        found_factors.add((min(p,q), max(p,q)))
        print(f'  測定値={measured_int:3d} → r={r} → {N} = {p} × {q} ✓')

if found_factors:
    print(f'\n最終結果: {N} = ', end='')
    for p, q in found_factors:
        print(f'{p} × {q}')

## 3. RSA への脅威の可視化

In [ ]:
# 古典 vs 量子の計算量比較
import numpy as np
import matplotlib.pyplot as plt

bit_lengths = np.arange(256, 4097, 256)

# 古典: 一般数体ふるい法 exp((64/9)^(1/3) * n^(1/3) * (log n)^(2/3))
def classical_complexity(n):
    """GNFS計算量の対数 (n: ビット長)"""
    return (64/9)**(1/3) * n**(1/3) * (np.log2(n))**(2/3)

# 量子: Shorアルゴリズム O(n^3) 論理演算
def quantum_complexity(n):
    """Shorアルゴリズムの量子ゲート数 (概算)"""
    return n**3  # O(n^3) logical operations

classical_ops = [classical_complexity(n) for n in bit_lengths]
quantum_ops = [np.log2(quantum_complexity(n)) for n in bit_lengths]
classical_ops_log = [np.log2(10) * c for c in classical_ops]  # log2 of operations

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 左: 計算量の比較
axes[0].plot(bit_lengths, classical_ops, 'r-o', label='古典 (GNFS): exp多項式', linewidth=2, markersize=4)
axes[0].plot(bit_lengths, quantum_ops, 'b-s', label='量子 (Shor): O(n³)', linewidth=2, markersize=4)
axes[0].set_xlabel('RSA鍵長 (ビット)', fontsize=12)
axes[0].set_ylabel('計算量指標 (対数スケール)', fontsize=12)
axes[0].set_title('RSA解読の計算量比較\n古典 vs 量子コンピュータ', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].fill_between(bit_lengths, classical_ops, quantum_ops, alpha=0.1, color='green',
                      label='量子優位性の領域')

# 右: 安全性タイムライン
scenarios = {
    'RSA-1024\n(非推奨)': {'classical': 2015, 'quantum': 2030},
    'RSA-2048\n(現在標準)': {'classical': 2050, 'quantum': 2035},
    'RSA-4096\n(高セキュリティ)': {'classical': 2080, 'quantum': 2040},
    'PQC-Kyber\n(次世代標準)': {'classical': 2100, 'quantum': 2100},
}

y_pos = np.arange(len(scenarios))
names = list(scenarios.keys())
classical_years = [v['classical'] for v in scenarios.values()]
quantum_years = [v['quantum'] for v in scenarios.values()]

axes[1].barh(y_pos - 0.2, [y - 2026 for y in classical_years], 0.35,
              left=2026, color='red', alpha=0.7, label='古典攻撃に対する安全年数')
axes[1].barh(y_pos + 0.2, [y - 2026 for y in quantum_years], 0.35,
              left=2026, color='blue', alpha=0.7, label='量子攻撃に対する安全年数')
axes[1].axvline(x=2026, color='black', linestyle='--', linewidth=1.5, label='現在 (2026)')
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(names, fontsize=10)
axes[1].set_xlabel('年', fontsize=12)
axes[1].set_title('暗号方式の安全性タイムライン\n(推定)', fontsize=13)
axes[1].legend(fontsize=9, loc='lower right')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('crypto_threat_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('脅威分析グラフを保存しました')

## 4. まとめ：PQCが必要な理由

| 暗号方式 | 古典攻撃 | 量子攻撃 | 状態 |
|----------|----------|----------|------|
| RSA-2048 | 指数時間 (安全) | **多項式時間 (破られる)** | 危険 |
| ECDSA-256 | 離散対数 (安全) | **Shorで解読可能** | 危険 |
| AES-256 | $2^{256}$ | Groverで $2^{128}$ (許容範囲) | 条件付き安全 |
| CRYSTALS-Kyber | 格子問題 (安全) | 量子耐性あり | **PQC標準** |
| CRYSTALS-Dilithium | 格子問題 (安全) | 量子耐性あり | **PQC標準** |

**次のノートブック**: `02_lattice_cryptography.ipynb` で格子ベース暗号の仕組みと実装を学びます。